# Notebook 08 · Retraining of the corrected RetinaNet (deviation 1, amendment v1.2)

**Why it exists:** in the first run, RetinaNet was trained with transposed boxes, because MONAI reads the
first box coordinate along the image rows. `ichdet` 0.4.0 corrects only this (fingerprint
`a6f3c0fc5798879d`); the YOLO arms and the evaluator did not change. Details: section 10 of
`docs/statistical_protocol.md` and `docs/osf_amendment_v1.2.md`.

**What it does:** the five RetinaNet jobs (50 epochs each, same folds and settings), each with the
prediction of its held-out fold. Output goes to `dados/tc/experimento/correcao_retinanet/` on the Drive;
the folder of the first run (`completo/`) is not touched. **No metric is computed here** and CT-ICH is not
used: the evaluation is done once, in notebook 09.

**Before running:**

1. Copy `src/ichdet` (**version 0.4.0**) of this repository to the Drive as
   `MyDrive/ich-benchmark/codigo/ichdet/`, replacing the older copy.
2. GPU: *Runtime → Change runtime type → L4* (high RAM).

**Time:** about 8 h per job, 41 h in total. With two sessions at once (`UMA_SESSAO = False` in both; for
example `RODADAS = [1, 2, 3]` in one and `[4, 5]` in the other), about 25 h; the Colab limit is 24 h per
session, so one of them must be opened again. Finished jobs are skipped and the interrupted one resumes
from the last saved epoch.

**Language:** documentation and comments are in English; identifiers, printed messages and the column
names of the files written here are in Portuguese, exactly as in the run that produced the results.

In [ ]:
# 1. Drive, libraries, GPU and check of the frozen code
import json, os, socket, subprocess, sys, time, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/ich-benchmark')
    LOCAL = Path('/content/cq500_512')               # Colab disk: reading many images from the Drive is slow
    from importlib.metadata import version
    numpy_antes = version('numpy')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'monai==1.5.2', 'ultralytics>=8.3,<8.4'],
                   check=True)
    if version('numpy') != numpy_antes:
        raise SystemExit(f'O pip trocou o numpy ({numpy_antes} -> {version("numpy")}). '
                         'Reinicie a sessão (Ambiente de execução → Reiniciar sessão) e rode de novo.')
except ImportError:                                  # local test: RAIZ mimics the Drive folder
    RAIZ = Path(os.environ['EXPERIMENTO_LOCAL'])
    LOCAL = RAIZ / 'cq500_512_local'

CODIGO = RAIZ / 'codigo'
sys.path.insert(0, str(CODIGO))
PREPARO = RAIZ / 'dados' / 'tc' / 'preparo'
SAIDA = RAIZ / 'dados' / 'tc' / 'experimento' / 'correcao_retinanet'
SAIDA.mkdir(parents=True, exist_ok=True)

import numpy as np, torch, monai, ultralytics
import ichdet
from ichdet import experiment, training_config as cfg

CONGELADO = 'a6f3c0fc5798879d'                             # fingerprint of the code of protocol v1.2 (corrected RetinaNet)
SESSAO = f'{socket.gethostname()}-{os.getpid()}'
AMBIENTE = {
    'ichdet_version': ichdet.__version__, 'frozen_code_fingerprint': experiment.frozen_fingerprint(CODIGO / 'ichdet'),
    'python': sys.version.split()[0], 'numpy': np.__version__, 'torch': torch.__version__, 'monai': monai.__version__,
    'ultralytics': ultralytics.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
    'session': SESSAO, 'started': time.strftime('%Y-%m-%d %H:%M:%S'),
}
print(json.dumps(AMBIENTE, indent=2))
assert AMBIENTE['frozen_code_fingerprint'] == CONGELADO, (
    'O código de treino no Drive não é o congelado. Copie de novo a pasta src/ichdet do repositório.')
assert torch.cuda.is_available() or 'EXPERIMENTO_LOCAL' in os.environ, 'Ative a GPU (Ambiente de execução → Alterar o tipo).'
with open(SAIDA / 'sessions.jsonl', 'a') as f:      # record of every session that ran
    f.write(json.dumps(AMBIENTE) + '\n')

In [ ]:
# 2. Which jobs this session runs; CQ500 images on the Colab disk; check of the five folds
RODADAS = [1, 2, 3, 4, 5]
BRACOS = ['retinanet']
UMA_SESSAO = True          # False if another session runs this notebook at the same time

if not (LOCAL / 'monai' / 'fold5.json').exists():
    inicio = time.time()
    with zipfile.ZipFile(PREPARO / 'cq500_512.zip') as z:
        z.extractall(LOCAL)
    print(f'descompactado em {time.time() - inicio:.0f} s')

esperado = json.loads((PREPARO / 'export_512_summary.json').read_text())['cq500_folds_main_analysis']
for k in experiment.FOLDS:
    d = json.loads((LOCAL / 'monai' / f'fold{k}.json').read_text())
    val, tre = {e['exam'] for e in d['validation']}, {e['exam'] for e in d['training']}
    assert len(d['validation']) == esperado[f'fold{k}']['slices'] and len(val) == esperado[f'fold{k}']['exams'], k
    assert not val & tre, f'rodada {k}: exame no treino e na validação'
    print(f'rodada {k}: treino {len(d["training"])} imagens; validação {len(d["validation"])} imagens de {len(val)} exames')

def quadro():
    linhas = experiment.status(SAIDA, arms=BRACOS, session=SESSAO)
    nomes = {'done': 'pronto', 'running': 'em outra sessão', 'partial': 'interrompido', 'pending': 'a fazer'}
    for l in linhas:
        print(f"rodada {l['fold']}  {l['arm']:15s}  {nomes[l['state']]:16s}  {l['epochs']:2d}/{cfg.EPOCHS} épocas")
    return linhas

_ = quadro()

In [ ]:
# 3. Corrected RetinaNet jobs (50 epochs + prediction of the held-out fold). Run again after disconnections.
for rodada, braco in experiment.job_order(RODADAS, BRACOS):
    experiment.run_job(rodada, braco, LOCAL / 'monai' / f'fold{rodada}.json', LOCAL, SAIDA, epochs=cfg.EPOCHS,
                       session=SESSAO, environment=AMBIENTE, take_over=UMA_SESSAO)

In [ ]:
# 4. Status of the five jobs and record of the run (in English)
linhas = quadro()
trabalhos = {}
for l in linhas:
    f = experiment.job_dir(SAIDA, l['fold'], l['arm']) / 'job.json'
    if f.exists():
        j = json.loads(f.read_text())
        trabalhos[f"fold{l['fold']}/{l['arm']}"] = {k: j[k] for k in (
            'epochs', 'training_images', 'validation_images', 'validation_exams', 'seconds_per_epoch',
            'prediction_seconds', 'boxes_predicted', 'finished', 'session')} | {'gpu': j['environment'].get('gpu')}
situacao = {'frozen_code_fingerprint': CONGELADO, 'jobs_done': len(trabalhos), 'jobs_total': len(linhas),
            'jobs': trabalhos}
(SAIDA / 'training_status.json').write_text(json.dumps(situacao, indent=2))
pronto = len(trabalhos) == len(linhas)
print(f"\n{len(trabalhos)} de {len(linhas)} treinos prontos." +
      (' Tudo pronto: siga para o notebook 09 (avaliação da correção).' if pronto else ' Rode de novo para continuar.'))

## After the five jobs

`training_status.json` and `sessions.jsonl` in `dados/tc/experimento/correcao_retinanet/` record the jobs
(included in this repository under `data/`). Weights and predictions stay on the Drive. Then run notebook
09 (evaluation of the correction, once).